# Phase 4 (Court Judgment Prediction, CJPE) — Run #3 on the Phase-1 **Encoder-Decoder model 01** (bidirectional encoder), with **IA³**

Config-driven like the other Phase-4 notebooks: edit **Cell 2** only. This is the **third** Phase-4 run of the plan: the **encoder** of the Phase-1 encoder-decoder, adapted with **IA³**, the lightest technique in the list. Data loading, chunking, the document model, metrics, threshold tuning, the submission writer and the run log are *identical* to the Dec-02 and Dec-01 notebooks, so the runs are directly comparable.

### The task in one paragraph
Each case is one long judgment (`{"id", "text"}`; train/dev also have a binary label). Predict **0 or 1** per case. Metric: **macro-F1**. The test file must give **exactly one integer prediction per test id**.

### Model = Phase-1 **encoder** per chunk → mean-pool → aggregator over chunks → `Linear(512 → 2)`
1. **Select** at most `MAX_CHUNKS` chunks of `CHUNK_TOKENS` tokens from the case (`DOC_VIEW`: `tail` / `head` / `head_tail` / `spread`).
2. **Encode** every chunk with the (adapted) encoder and **mean-pool** its tokens into `h_c`.
3. **Aggregate** the chunk vectors (`AGG`: `mean`, `attn` = `α_c = softmax(vᵀ tanh(W h_c))`, or `hier` = a small Transformer over the chunk vectors).
4. `dropout → Linear(512 → 2)`, class-weighted cross-entropy, decision threshold tuned on dev.

Cost: attention over a whole case of `K·L` tokens would be `O((KL)²)`; independent chunks cost `O(K·L²)`.

### Why this model is the natural fit for classifying a chunk
| | Decoders 01 / 02 | **Encoder (this notebook)** |
|---|---|---|
| context for a token | left only (Dec-01) / unmasked by a patch (Dec-02) | **both sides, natively**: full self-attention (`encoder_causal=False`) |
| Phase-1 training signal | next-token prediction | **masked-LM** (15 %, 80/10/10) on the encoder, plus the seq2seq loss through cross-attention |
| used at fine-tuning time | the whole model | **the encoder only**; decoder, cross-attention and LM head are dropped |

Size: embeddings 16.38M + 6 encoder blocks 19.27M = **35.66M** (the full Phase-1 model is 61.49M). The encoder has 6 layers, so per token it is roughly half the cost of a 12-layer decoder. **KD warning:** the teacher must be *larger* than the student and 35.66M is below the default 36.58M Dec-03 student, so this run **cannot** be that student's teacher.

The encoder was trained on 768-token inputs (`encoder_len`); its RoPE table covers 1024 positions. `CHUNK_TOKENS=512` (the same as the decoder runs, for comparability) is inside the trained length; the notebook warns above 768.

### IA³ — *Infused Adapter by Inhibiting and Amplifying Inner Activations* (Liu et al. 2022)
IA³ freezes the whole backbone and learns **one scaling vector per site**, initialised to 1 (so the model starts exactly at the Phase-1 function; asserted at build time):

* `k ← l_k ⊙ (W_k x)` and `v ← l_v ⊙ (W_v x)` in every attention layer (`l_k, l_v ∈ R^512`),
* `h_ff ← l_ff ⊙ (silu(W_gate x) ⊙ W_up x)` before `down_proj` in every SwiGLU block (`l_ff ∈ R^1408`).

That is `512 + 512 + 1408 = 2 432` numbers per block, **14 592** for the six encoder blocks. LoRA at r = 16 on the 7 linear types of the same six blocks has about 0.95M. IA³ is *diagonal* (it can only amplify or inhibit existing feature channels), LoRA adds a dense low-rank update, LoRMA multiplies the layer output by a low-rank-plus-identity matrix. So expect IA³ to behave like "a probe on the pretrained features plus a per-channel re-weighting"; whether the MLM-pretrained encoder features are already good enough for judgment prediction is exactly the capstone's question *"are the Phase-1 representations useful?"*. The larger learning rate (3e-3, the IA³ paper's setting) is deliberate: the vectors start at 1 and only need small moves.

**Honest parameter accounting for the report.** The trainable count printed at build time is IA³ (14.6k) **plus the document head**: `AGG="attn"` adds 0.26M (`W`, `v`) and the classifier 1k. The *PEFT* part is the 14.6k; if you want the purest IA³ number, run `AGG="mean"` (head only, 1k) as a control.

### `METHOD`
| `METHOD` | What is trained |
|---|---|
| `"ia3"` | the IA³ vectors at the sites in `IA3_SITES` + aggregator/head (**default**) |
| `"full"` | encoder + embeddings + aggregator/head (full-fine-tuning reference) |
| `"lora"` / `"lorma"` | adapters on the 7 Linear types of every encoder block + aggregator/head (same code as the decoder notebooks) |
| `"norm_only"` | RMSNorm gains + aggregator/head |

### What this notebook writes
`submission.zip` (verified like the evaluator would), `run_log.json` (parameters, peak GPU memory, train time, inference ms per case, dev metrics, per-class P/R/F1, confusion matrix, per-slot attention profile, FP/FN by length bucket), `dev_errors.json`.

In [1]:
# Cell 0 — Hugging Face login (token from Kaggle secrets), same as the Phase-1 notebooks
import os
try:
    from huggingface_hub import login
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF")
    os.environ["HF_TOKEN"] = _tok
    login(_tok)
    print("HF login ok")
except Exception as e:
    print("HF secret not available:", e)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login ok


In [2]:
# Cell 1 — Imports, paths, device, precision (same conventions as the Phase-1 notebooks)
import os, sys, json, math, time, random, gc, zipfile, types, re, pickle
from pathlib import Path
from dataclasses import dataclass, asdict, field, fields
from typing import Optional, Dict, List, Tuple, Any
from collections import Counter, defaultdict

import multiprocessing as mp
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

IS_KAGGLE = Path("/kaggle/working").exists()
WORK_DIR = Path("/kaggle/working") if IS_KAGGLE else Path("./work")
WORK_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR = Path("/kaggle/input/datasets/ascyrax/capstone-tokenizer-large") if IS_KAGGLE else Path("./input")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    cap = torch.cuda.get_device_capability(0)
    AMP_DTYPE = torch.bfloat16 if cap[0] >= 8 else torch.float16     # T4/P100 -> fp16 + GradScaler
    USE_AMP = True
    print("GPU:", torch.cuda.get_device_name(0), "| capability", cap)
else:
    AMP_DTYPE, USE_AMP = torch.float32, False
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16
print(f"device={device} | AMP dtype={AMP_DTYPE} | grad scaler={USE_SCALER} | torch {torch.__version__}")

SEED = 1337
def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if device.type == "cuda": torch.cuda.manual_seed_all(s)
seed_everything(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

GPU: Tesla T4 | capability (7, 5)
device=cuda | AMP dtype=torch.float16 | grad scaler=True | torch 2.11.0+cu128


In [3]:
# Cell 2 — RUN CONFIG  (the only cell you normally edit)

RUN_NAME   = "p4_encdec_ia3"
METHOD     = "ia3"            # "ia3" | "lora" | "lorma" | "full" | "norm_only"
SMOKE_TEST = False            # True: ~150 steps on a small random subset, dev check only, NO submission written
ATTN_MODE  = "bidir"          # FIXED: the encoder is bidirectional by construction (kept as a constant for the shared cells)
IA3_SITES  = ("k", "v", "ff") # METHOD="ia3": which activations get a learned scaling vector (keys, values, FFN hidden)

# ---- Phase-1 checkpoint (written by caps_enc_dec_architecture_01.ipynb; must have encoder_causal=False) ----------------------------
CKPT_PATH = Path("/kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt") if IS_KAGGLE else Path("./input/best.pt")

# ---- Data ---------------------------------------------------------------------------------
HF_REPO = "Exploration-Lab/CS-781-Capstone"      # files are Phase_4_CJPE/{Train/train.json, Train/dev.json, Test/test.json}
LOCAL_DATA_DIR = None                             # or a folder that contains Phase_4_CJPE/ (checked first)
FINAL_FIT  = False                                # True: train on train+dev, NO validation (use after the dev run, EPOCHS = best epoch)

# ---- Long-document representation -----------------------------------------------------------
DOC_VIEW     = "tail"        # "tail" | "head" | "head_tail" | "spread"   (see the first markdown cell)
CHUNK_TOKENS = 512           # tokens per chunk INCLUDING the leading [BOS]  (<= 1024 = RoPE table; the encoder was TRAINED on 768)
MAX_CHUNKS   = 4             # chunks per case  => at most MAX_CHUNKS*(CHUNK_TOKENS-1) tokens are read
AGG          = "attn"        # "mean" | "attn" | "hier"
HIER_LAYERS  = 1             # AGG="hier": number of Transformer layers over the chunk vectors
CHAR_PER_TOK = 8             # tokenisation shortcut for long texts: only a slice of CHAR_PER_TOK chars per wanted token is tokenised
N_PROC       = 4             # worker processes for tokenisation (1 = serial)

# ---- Training -----------------------------------------------------------------------------
EPOCHS       = 6                          # IA3 trains few parameters and converges slowly (the Phase-3 run used 8); the 6-layer encoder is ~2x faster per epoch than the 12-layer decoders
BATCH_DOCS   = 4             # cases per micro-batch (each case = up to MAX_CHUNKS sequences)
ACCUM        = 4             # gradient accumulation  => effective batch = BATCH_DOCS*ACCUM cases
WARMUP_FRAC  = 0.06
GRAD_CLIP    = 1.0
GRAD_CKPT    = False         # True: recompute block activations in the backward pass (less memory, ~30% slower)
TIME_BUDGET_MIN = None       # e.g. 150: stop training after this many minutes (the best epoch so far is kept)
HP = {                                            # per-method hyper-parameters (starting points, tune on dev)
    "ia3":       dict(lr=3e-3, head_lr=1e-3, wd=0.0),
    "lorma":     dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, variant="plus"),   # variant: "plus" (LoRMA+) | "pi" (LoRMA_pi)
    "lora":      dict(lr=5e-4, head_lr=1e-3, wd=0.0, r=16, alpha=32, dropout=0.05),
    "full":      dict(lr=1e-4, head_lr=1e-3, wd=0.01),
    "norm_only": dict(lr=1e-3, head_lr=1e-3, wd=0.0),
}
ADAPTER_TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")   # LoRA and LoRMA
HEAD_DROPOUT    = 0.1
CLASS_WEIGHT    = "sqrt_inv"                      # "sqrt_inv" | "none"
MAX_CLASS_W     = 10.0
LABEL_SMOOTHING = 0.0

# ---- Decision threshold tuned on dev (macro-F1 post-processing) -----------------------------
THRESH_TUNE     = True
THRESH_MIN_GAIN = 0.005                           # keep the tuned threshold only if a 2-fold held-out check on dev gains this much macro-F1
THRESH_FINAL    = None                            # used when FINAL_FIT=True: paste the bias printed by the dev run (or None)

# ---- Smoke test -----------------------------------------------------------------------------
SMOKE_STEPS, SMOKE_TRAIN_DOCS, SMOKE_VAL_DOCS = 150, 800, 200

# ---- Knowledge distillation: only useful for a student smaller than 35.66M --------------------------------------------------
EXPORT_SOFT_LABELS = False        # this encoder (35.66M) is SMALLER than the default 36.58M student, so it cannot be its teacher

# ---- derived ----------------------------------------------------------------------------------
assert METHOD in HP and ATTN_MODE == "bidir" and set(IA3_SITES) <= {"k", "v", "ff"} and len(IA3_SITES) > 0
assert DOC_VIEW in ("tail", "head", "head_tail", "spread") and AGG in ("mean", "attn", "hier")
assert 16 <= CHUNK_TOKENS <= 1024 and MAX_CHUNKS >= 1, "chunks longer than 1024 exceed the Phase-1 RoPE table"
BODY = CHUNK_TOKENS - 1                           # tokens of text per chunk (the first position is [BOS])
SEQ_LEN = CHUNK_TOKENS                            # (used by the checkpoint-loading cell)
RUN_DIR = WORK_DIR / "runs" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"RUN {RUN_NAME} | METHOD={METHOD} | encoder-only (bidirectional) | view={DOC_VIEW} {MAX_CHUNKS}x{CHUNK_TOKENS} | agg={AGG} | "
      f"batch={BATCH_DOCS}x{ACCUM} | epochs={EPOCHS} | final_fit={FINAL_FIT} | smoke={SMOKE_TEST}")

RUN p4_encdec_ia3 | METHOD=ia3 | encoder-only (bidirectional) | view=tail 4x512 | agg=attn | batch=4x4 | epochs=6 | final_fit=False | smoke=False


In [4]:
# Cell 3 — The shared byte-level BPE tokenizer (same as every Phase-1 model)
if str(TOKENIZER_DIR) not in sys.path:
    sys.path.append(str(TOKENIZER_DIR))
from bpe_tokenizer import BPETokenizer, load_tokenizer

tokenizer = load_tokenizer(str(TOKENIZER_DIR))
VOCAB_SIZE = len(tokenizer.id_to_bytes)
BOS_ID, EOS_ID, PAD_ID, UNK_ID = tokenizer.bos_id, tokenizer.eos_id, tokenizer.pad_id, tokenizer.unk_id
print("vocab_size:", VOCAB_SIZE, "| BOS", BOS_ID, "EOS", EOS_ID, "PAD", PAD_ID)

# Sentences are tokenised exactly as given (no clean_text / normalisation).
_s = "Hon'ble Mr. Justice R. K. Gauba — § 302 IPC, ₹ 5,000 “quoted”."
_ids, _offs = tokenizer.encode(_s, return_offsets=True)
assert tokenizer.decode(_ids) == _s, "tokenizer round-trip failed"
assert all(0 <= a <= b <= len(_s) for a, b in _offs) and _offs[0][0] == 0 and _offs[-1][1] == len(_s)
print("offset sanity ok |", len(_ids), "tokens for", len(_s), "chars")

# Guard against O(L^2) BPE on pathological chunks (long runs of dashes, dots, underscores, spaces) -- the same guard the Phase-1
# Decoder-03 notebook uses. Pre-tokens longer than 1000 bytes are BPE-encoded in 1000-byte pieces. Normal text never has such
# chunks, so its tokenisation (and every character offset) is unchanged; byte-level BPE decodes the pieces to the exact same bytes.
_MAX_PRETOK_BYTES = 1000
_orig_encode_pretok = tokenizer._encode_pretok_bytes
def _encode_pretok_bytes_capped(bs):
    if len(bs) <= _MAX_PRETOK_BYTES:
        return _orig_encode_pretok(bs)
    out = []
    for i in range(0, len(bs), _MAX_PRETOK_BYTES):
        out.extend(_orig_encode_pretok(bs[i:i + _MAX_PRETOK_BYTES]))
    return out
tokenizer._encode_pretok_bytes = _encode_pretok_bytes_capped
_t0 = time.time(); _ids = tokenizer.encode("-" * 200_000)
assert tokenizer.decode(_ids) == "-" * 200_000
print(f"long-run guard ok: 200k-char run encoded in {time.time() - _t0:.1f}s")

vocab_size: 32000 | BOS 2 EOS 3 PAD 0
offset sanity ok | 28 tokens for 62 chars
long-run guard ok: 200k-char run encoded in 0.1s


## Phase-1 model code — copied **verbatim** from `caps_enc_dec_architecture_01.ipynb` (Cell 3)
Do not edit: the checkpoint only loads correctly if these classes are identical.

In [5]:
# ===================== Cell 3: model =====================
@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_encoder_layers: int = 6
    n_decoder_layers: int = 6
    d_model: int = 512
    n_head: int = 8
    n_kv_head: int = 8
    head_dim: int = 64
    ff_mult: float = 8 / 3
    d_ff: Optional[int] = None
    max_seq_len: int = 1024          # RoPE table size; must cover encoder_len and decoder_len
    encoder_len: int = 768
    decoder_len: int = 256
    norm_type: str = "rmsnorm"
    norm_eps: float = 1e-5
    pos_type: str = "rope"           # rope | learned
    rope_theta: float = 10000.0
    mlp_type: str = "swiglu"         # swiglu | gelu
    bias: bool = False
    attn_dropout: float = 0.0
    resid_dropout: float = 0.0
    embd_dropout: float = 0.0
    tie_embeddings: bool = True
    init_std: float = 0.02
    use_sdpa: bool = True
    encoder_causal: bool = False     # False = bidirectional encoder
    mlm_head: bool = True            # BERT-style MLM head on the encoder (auxiliary objective)
    arch_name: str = "encdec_bidir_mlm_v2"

    def __post_init__(self):
        assert self.d_model == self.n_head * self.head_dim
        assert self.n_head % self.n_kv_head == 0
        assert self.max_seq_len >= max(self.encoder_len, self.decoder_len)
        if self.d_ff is None:
            raw = int(self.ff_mult * self.d_model)
            self.d_ff = 64 * ((raw + 63) // 64)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        rms = torch.sqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return (x.float() / rms).to(x.dtype) * self.g

class LayerNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.g = nn.Parameter(torch.ones(d)); self.b = nn.Parameter(torch.zeros(d)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        mean = xf.mean(-1, keepdim=True)
        var = xf.var(-1, keepdim=True, unbiased=False)
        return ((xf - mean) / torch.sqrt(var + self.eps)).to(x.dtype) * self.g + self.b

def build_norm(cfg):
    if cfg.norm_type == "rmsnorm":
        return RMSNorm(cfg.d_model, cfg.norm_eps)
    if cfg.norm_type == "layernorm":
        return LayerNorm(cfg.d_model, cfg.norm_eps)
    raise ValueError(cfg.norm_type)


class RoPE(nn.Module):
    def __init__(self, head_dim, max_seq_len, theta=10000.0):
        super().__init__()
        assert head_dim % 2 == 0
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        freqs = torch.outer(torch.arange(max_seq_len).float(), inv_freq)
        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    @staticmethod
    def _rotate(x, cos, sin):
        x1, x2 = x[..., 0::2], x[..., 1::2]
        return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

    def forward(self, q, k, position_ids=None):
        T = q.shape[2]
        if position_ids is None:
            cos = self.cos[:T][None, None]; sin = self.sin[:T][None, None]
        else:                                    # position_ids: [B, T] or [1, T]
            cos = self.cos[position_ids].unsqueeze(1); sin = self.sin[position_ids].unsqueeze(1)
        dt = q.dtype
        q = self._rotate(q.float(), cos, sin); k = self._rotate(k.float(), cos, sin)
        return q.to(dt), k.to(dt)


class Attention(nn.Module):
    # Self-attention (optionally causal, with KV cache) or cross-attention (context / precomputed static_kv).
    def __init__(self, cfg, causal):
        super().__init__()
        self.cfg, self.causal = cfg, causal
        self.n_head, self.n_kv_head, self.head_dim = cfg.n_head, cfg.n_kv_head, cfg.head_dim
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.q_proj = nn.Linear(cfg.d_model, cfg.n_head * cfg.head_dim, bias=cfg.bias)
        self.k_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.v_proj = nn.Linear(cfg.d_model, cfg.n_kv_head * cfg.head_dim, bias=cfg.bias)
        self.o_proj = nn.Linear(cfg.n_head * cfg.head_dim, cfg.d_model, bias=cfg.bias)
        self.resid_dropout = nn.Dropout(cfg.resid_dropout)

    def _split(self, x, nh):
        B, T, _ = x.shape
        return x.view(B, T, nh, self.head_dim).transpose(1, 2)

    def project_kv(self, src):
        return self._split(self.k_proj(src), self.n_kv_head), self._split(self.v_proj(src), self.n_kv_head)

    def _attend(self, q, k, v, causal, key_mask):
        T, S = q.size(2), k.size(2)
        dp = self.cfg.attn_dropout if self.training else 0.0
        mask, sdpa_causal = None, False
        if causal and T > 1:
            if T == S and key_mask is None:
                sdpa_causal = True
            else:   # general (offset) causal mask: query i sees keys <= i + (S - T)
                mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril(diagonal=S - T).view(1, 1, T, S)
        if key_mask is not None:                 # key_mask: [B, S], True/1 = real token
            km = key_mask.to(torch.bool).view(key_mask.size(0), 1, 1, S)
            mask = km if mask is None else (mask & km)
        if self.cfg.use_sdpa:
            return F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=dp, is_causal=sdpa_causal)
        scores = (q.float() @ k.float().transpose(-2, -1)) * self.scale
        if sdpa_causal:
            mask = torch.ones(T, S, dtype=torch.bool, device=q.device).tril().view(1, 1, T, S)
        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))
        p = F.dropout(F.softmax(scores, dim=-1), dp, self.training).to(q.dtype)
        return p @ v

    def forward(self, x, rope=None, position_ids=None, key_mask=None, kv_cache=None,
                context=None, static_kv=None):
        B, T, _ = x.shape
        q = self._split(self.q_proj(x), self.n_head)
        is_cross = (context is not None) or (static_kv is not None)
        if static_kv is not None:
            k, v = static_kv
        elif context is not None:
            k, v = self.project_kv(context)
        else:
            k, v = self.project_kv(x)
            if rope is not None:
                q, k = rope(q, k, position_ids=position_ids)
            if kv_cache is not None:
                k = torch.cat([kv_cache[0], k], dim=2)
                v = torch.cat([kv_cache[1], v], dim=2)
        new_cache = None if is_cross else (k, v)
        if self.n_kv_head != self.n_head:
            rep = self.n_head // self.n_kv_head
            k = k.repeat_interleave(rep, dim=1); v = v.repeat_interleave(rep, dim=1)
        out = self._attend(q, k, v, causal=(self.causal and not is_cross), key_mask=key_mask)
        out = out.transpose(1, 2).contiguous().view(B, T, -1)
        return self.resid_dropout(self.o_proj(out)), new_cache


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.gate_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.up_proj = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))

class GELUMLP(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg.d_model, cfg.d_ff, bias=cfg.bias)
        self.down_proj = nn.Linear(cfg.d_ff, cfg.d_model, bias=cfg.bias)
        self.drop = nn.Dropout(cfg.resid_dropout)
    def forward(self, x):
        return self.drop(self.down_proj(F.gelu(self.fc1(x))))

def build_mlp(cfg):
    return SwiGLU(cfg) if cfg.mlp_type == "swiglu" else GELUMLP(cfg)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=cfg.encoder_causal)
        self.norm2, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, rope=None, key_mask=None, position_ids=None):
        h, _ = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, key_mask=key_mask)
        x = x + h
        return x + self.mlp(self.norm2(x))

class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1, self.self_attn = build_norm(cfg), Attention(cfg, causal=True)
        self.norm2, self.cross_attn = build_norm(cfg), Attention(cfg, causal=False)
        self.norm3, self.mlp = build_norm(cfg), build_mlp(cfg)
    def forward(self, x, enc_out=None, cross_kv=None, enc_mask=None, rope=None,
                position_ids=None, kv_cache=None):
        h, new_self = self.self_attn(self.norm1(x), rope=rope, position_ids=position_ids, kv_cache=kv_cache)
        x = x + h
        if cross_kv is not None:
            h, _ = self.cross_attn(self.norm2(x), static_kv=cross_kv, key_mask=enc_mask)
        else:
            h, _ = self.cross_attn(self.norm2(x), context=enc_out, key_mask=enc_mask)
        x = x + h
        return x + self.mlp(self.norm3(x)), new_self


class Encoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.n_encoder_layers)])
        self.norm_f = build_norm(cfg)
    def forward(self, input_ids, attention_mask=None, position_ids=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        for blk in self.blocks:
            x = blk(x, rope=self.rope, key_mask=attention_mask, position_ids=position_ids)
        return self.norm_f(x)

class Decoder(nn.Module):
    def __init__(self, cfg, tok_emb):
        super().__init__()
        self.cfg, self.tok_emb = cfg, tok_emb
        self.emb_drop = nn.Dropout(cfg.embd_dropout)
        self.rope = RoPE(cfg.head_dim, cfg.max_seq_len, cfg.rope_theta) if cfg.pos_type == "rope" else None
        self.pos_emb = nn.Embedding(cfg.max_seq_len, cfg.d_model) if cfg.pos_type == "learned" else None
        self.blocks = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.n_decoder_layers)])
        self.norm_f = build_norm(cfg)

    def precompute_cross_kv(self, enc_out):
        # Cross-attention K/V depend only on the encoder output: compute once per encoder pass.
        return [blk.cross_attn.project_kv(enc_out) for blk in self.blocks]

    def forward(self, input_ids, enc_out=None, enc_mask=None, position_ids=None,
                kv_caches=None, cross_kvs=None):
        x = self.tok_emb(input_ids)
        if self.pos_emb is not None:
            if position_ids is None:
                position_ids = torch.arange(input_ids.size(1), device=input_ids.device)[None]
            x = x + self.pos_emb(position_ids)
        x = self.emb_drop(x)
        new_caches = []
        for i, blk in enumerate(self.blocks):
            x, nc = blk(x, enc_out=enc_out,
                        cross_kv=cross_kvs[i] if cross_kvs is not None else None,
                        enc_mask=enc_mask, rope=self.rope, position_ids=position_ids,
                        kv_cache=kv_caches[i] if kv_caches is not None else None)
            new_caches.append(nc)
        return self.norm_f(x), new_caches


class Seq2SeqLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.encoder = Encoder(cfg, self.tok_emb)
        self.decoder = Decoder(cfg, self.tok_emb)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        if cfg.tie_embeddings:
            self.lm_head.weight = self.tok_emb.weight
        # BERT-style transform before the (tied) output projection for MLM.
        self.mlm_transform = (nn.Sequential(nn.Linear(cfg.d_model, cfg.d_model, bias=True), nn.GELU(),
                                            build_norm(cfg)) if cfg.mlm_head else None)

    def encode(self, enc_ids, enc_mask=None):
        return self.encoder(enc_ids, attention_mask=enc_mask)

    def forward(self, enc_ids, dec_ids, enc_mask=None):
        enc_out = self.encode(enc_ids, enc_mask)
        h, _ = self.decoder(dec_ids, enc_out=enc_out, enc_mask=enc_mask)
        return self.lm_head(h)

    def mlm_logits_at(self, enc_hidden, select):
        # Project only the masked positions -> [N, V] (avoids a full [B, T, V] tensor).
        return self.lm_head(self.mlm_transform(enc_hidden[select]))


def init_weights(model, cfg):
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, cfg.init_std)
    # GPT-2-style residual scaling, per stack: encoder blocks add 2 residual branches, decoder blocks add 3.
    s_enc = 1.0 / math.sqrt(2 * cfg.n_encoder_layers)
    s_dec = 1.0 / math.sqrt(3 * cfg.n_decoder_layers)
    with torch.no_grad():
        for blk in model.encoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_enc); blk.mlp.down_proj.weight.mul_(s_enc)
        for blk in model.decoder.blocks:
            blk.self_attn.o_proj.weight.mul_(s_dec); blk.cross_attn.o_proj.weight.mul_(s_dec)
            blk.mlp.down_proj.weight.mul_(s_dec)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())   # .parameters() de-duplicates tied weights
    emb = model.tok_emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


# ---- Checkpoint helpers (with the causal/bidirectional guard) ----
def cfg_from_dict(d):
    d = dict(d)
    if "encoder_causal" not in d:
        raise ValueError("Checkpoint config has no 'encoder_causal' key. This is probably an old causal-encoder "
                         "checkpoint. Refusing to guess: the flag is a mask, not a weight, so a wrong guess would "
                         "load silently and run the encoder in a mode it was never trained in. Add "
                         "ckpt['config']['encoder_causal'] = True/False manually if you know which run it is.")
    d.setdefault("mlm_head", False)          # pre-v2 checkpoints have no MLM head
    d.setdefault("arch_name", "legacy")
    known = {f.name for f in fields(ModelConfig)}
    dropped = sorted(set(d) - known)
    if dropped:
        print("cfg_from_dict: ignoring unknown keys", dropped)
    return ModelConfig(**{k: v for k, v in d.items() if k in known})

def atomic_save(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_model_from_ckpt(path, expect_encoder_causal=False):
    ck = torch.load(path, map_location="cpu", weights_only=False)
    cfg = cfg_from_dict(ck["config"])
    if expect_encoder_causal is not None and cfg.encoder_causal != expect_encoder_causal:
        raise ValueError(f"{path}: encoder_causal={cfg.encoder_causal}, expected {expect_encoder_causal}")
    model = Seq2SeqLM(cfg)
    model.load_state_dict(ck["model"], strict=True)
    st = ck.get("state", {})
    print(f"Loaded {path} | arch={cfg.arch_name} | encoder_causal={cfg.encoder_causal} | "
          f"enc_len={cfg.encoder_len} dec_len={cfg.decoder_len} | step={st.get('step', ck.get('step'))} | "
          f"val_loss={st.get('val_loss', ck.get('val_loss'))}")
    return model.to(device).eval(), cfg, ck

_c = ModelConfig(vocab_size=VOCAB_SIZE)
_pc = count_params(Seq2SeqLM(_c))
print(f"Default config params: total={_pc['total']/1e6:.2f}M (embedding {_pc['embedding']/1e6:.2f}M)")
del _c

Default config params: total=61.49M (embedding 16.38M)


## Load the Phase-1 checkpoint

In [6]:
# Cell 12 — Load the Phase-1 encoder-decoder checkpoint and sanity-check its ENCODER
lm, lm_cfg, _ck = load_model_from_ckpt(CKPT_PATH, expect_encoder_causal=False)       # refuses a causal-encoder checkpoint
del _ck
pc = count_params(lm)
n_enc = sum(p.numel() for p in lm.encoder.parameters())                              # embeddings (shared) + encoder blocks + final norm
print(f"Phase-1 model | arch={lm_cfg.arch_name} | enc layers={lm_cfg.n_encoder_layers} dec layers={lm_cfg.n_decoder_layers} "
      f"d_model={lm_cfg.d_model} pos={lm_cfg.pos_type} | total {pc['total']/1e6:.2f}M | ENCODER-ONLY (used for CJPE) {n_enc/1e6:.2f}M")
assert SEQ_LEN <= lm_cfg.max_seq_len, "window longer than the RoPE table of this checkpoint"
if SEQ_LEN > lm_cfg.encoder_len:
    print(f"WARNING: sequence length {SEQ_LEN} > the encoder's trained length {lm_cfg.encoder_len} (RoPE extrapolation, expect some degradation)")

# Sanity: if the weights did not load properly, masked-token prediction would sit near ln(V) ~ 10.4.
MASK_ID = getattr(tokenizer, "special_to_id", {}).get("<mask>", 6)
if lm_cfg.mlm_head:
    _sent = "The appeal is allowed and the impugned judgment of the High Court is set aside."
    _x = torch.tensor([[BOS_ID] + tokenizer.encode(_sent)], device=device)
    _pos = [p for p in (4, 9, 13) if p < _x.size(1) - 1]
    _xm = _x.clone(); _xm[0, _pos] = MASK_ID
    _sel = torch.zeros_like(_xm, dtype=torch.bool); _sel[0, _pos] = True
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
        _h = lm.encode(_xm, torch.ones_like(_xm))
        _lg = lm.mlm_logits_at(_h, _sel)
    _l = F.cross_entropy(_lg.float(), _x[0, _pos]).item()
    print(f"masked-LM loss on a legal sentence (3 masked tokens): {_l:.3f}  (ln V = {math.log(VOCAB_SIZE):.2f})")
    assert _l < 9.0, "checkpoint looks untrained / mis-loaded"
else:
    print("this checkpoint has no MLM head; skipping the masked-LM sanity check")

Loaded /kaggle/input/datasets/ascyrax/caps-encdec-01-best/best.pt | arch=encdec_bidir_mlm_v2 | encoder_causal=False | enc_len=768 dec_len=256 | step=15488 | val_loss=1.9592824522405863
Phase-1 model | arch=encdec_bidir_mlm_v2 | enc layers=6 dec layers=6 d_model=512 pos=rope | total 61.49M | ENCODER-ONLY (used for CJPE) 35.66M
masked-LM loss on a legal sentence (3 masked tokens): 0.117  (ln V = 10.37)


## Phase-4 data: cases, labels, statistics
Cases are `{"id", "text"}` (+ `"label"` for train/dev). The loader accepts a few equivalent layouts (a list, or a dict keyed by id; `text` may be a string or a list of strings; the label key may be `label`/`labels`/`y`/`judgment`/`outcome`), and **fails loudly** if a label is not 0/1. The statistics cell tells you how much of a typical case your `MAX_CHUNKS × CHUNK_TOKENS` budget covers. If the schema printed below is not what you expect, paste it to Claude.

In [7]:
# Cell 13 — Phase-4 data: train / dev / test cases
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

P4 = "Phase_4_CJPE"

def fetch(rel: str) -> Path:
    if LOCAL_DATA_DIR is not None and (Path(LOCAL_DATA_DIR) / rel).exists():
        return Path(LOCAL_DATA_DIR) / rel
    try:
        return Path(hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=rel, token=os.environ.get("HF_TOKEN")))
    except Exception as ex:
        raise RuntimeError(f"could not download '{rel}' from {HF_REPO} ({type(ex).__name__}: {ex}). "
                           f"Check the HF login (Cell 0) / internet setting, or download the Phase_4_CJPE folder and set LOCAL_DATA_DIR.") from ex

def _first(e, keys, default=None):
    for k in keys:
        if k in e: return e[k]
    return default

def parse_cases(raw, split, has_labels):
    """-> list of {uid, id, text, y|None, nchar}.  uid = '<split>::<position in the file>' (stable, used internally and by the KD export)."""
    if isinstance(raw, dict):
        if isinstance(raw.get("data"), list): items = raw["data"]
        else:
            items = []
            for k, v in raw.items():
                v = dict(v) if isinstance(v, dict) else {"text": v}
                v.setdefault("id", k); items.append(v)
    else:
        items = list(raw)
    docs = []
    for i, e in enumerate(items):
        rid = _first(e, ("id", "case_id", "doc_id", "document_id"), i)
        text = _first(e, ("text", "case_text", "document", "judgment_text", "content"))
        if isinstance(text, list): text = "\n".join(str(t) for t in text)
        if not isinstance(text, str):
            raise ValueError(f"{split} case {rid}: cannot find the case text. Keys: {list(e)}")
        y = None
        if has_labels:
            lab = _first(e, ("label", "labels", "y", "judgment", "outcome", "decision"))
            try: y = int(lab)
            except Exception: raise ValueError(f"{split} case {rid}: label {lab!r} is not 0/1. Keys: {list(e)}")
            if y not in (0, 1): raise ValueError(f"{split} case {rid}: label {y} is not 0/1")
        docs.append(dict(uid=f"{split}::{i}", id=rid, text=text, y=y, nchar=len(text)))
    return docs

def load_split(rel, split, has_labels):
    raw = json.loads(fetch(rel).read_text(encoding="utf-8"))
    first = raw[0] if isinstance(raw, list) and raw else (next(iter(raw.values())) if isinstance(raw, dict) and raw else None)
    print(f"  {rel}: {type(raw).__name__} of {len(raw)} | first item keys: {list(first) if isinstance(first, dict) else type(first).__name__}")
    return parse_cases(raw, split, has_labels)

TRAIN_ALL = load_split(f"{P4}/Train/train.json", "train", True)
DEV_ALL   = load_split(f"{P4}/Train/dev.json",   "dev",   True)
TEST_ALL  = load_split(f"{P4}/Test/test.json",   "test",  False)

for name, ds in (("train", TRAIN_ALL), ("dev", DEV_ALL), ("test", TEST_ALL)):
    nc = np.asarray([d["nchar"] for d in ds])
    msg = f"{name:5s}: {len(ds):6d} cases | chars/case mean {nc.mean():9.0f} median {np.median(nc):9.0f} p90 {np.percentile(nc, 90):9.0f} max {nc.max():9d}"
    if ds[0]["y"] is not None: msg += f" | label counts {dict(sorted(Counter(d['y'] for d in ds).items()))}"
    print(msg)
for name, ds in (("dev", DEV_ALL), ("test", TEST_ALL)):
    ids = [str(d["id"]) for d in ds]
    if len(set(ids)) != len(ids): print(f"WARNING: {name} ids are not unique -> paste this to Claude (the submission needs one prediction per id)")
_tr_ids, _te_ids = {str(d["id"]) for d in TRAIN_ALL}, {str(d["id"]) for d in TEST_ALL}
print("train/test id overlap:", len(_tr_ids & _te_ids))

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/train.json: list of 37387 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Train/dev.json: list of 1988 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']


Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

  Phase_4_CJPE/Test/test.json: list of 1517 | first item keys: ['id', 'text', 'label', 'expert_1', 'expert_2', 'expert_3', 'expert_4', 'expert_5']
train:  37387 cases | chars/case mean     19531 median     13023 p90     38996 max   2492701 | label counts {0: 22067, 1: 15320}
dev  :   1988 cases | chars/case mean     22110 median     16700 p90     40495 max    260198 | label counts {0: 994, 1: 994}
test :   1517 cases | chars/case mean     22698 median     17320 p90     42859 max    233000
train/test id overlap: 0


In [8]:
# Cell 14 — Length statistics: how much of a case does the chunk budget cover? (tokenises a small random sample of full texts)
_rng = random.Random(SEED)
_sample = _rng.sample(TRAIN_ALL, min(150, len(TRAIN_ALL)))
_t0 = time.time()
_tok_lens = np.asarray([len(tokenizer.encode(d["text"])) for d in _sample])
_dt = time.time() - _t0
_chars = np.asarray([d["nchar"] for d in _sample])
print(f"sample of {len(_sample)} train cases | chars/token {(_chars.sum() / _tok_lens.sum()):.2f} | full-text tokenisation {_chars.sum() / _dt / 1e6:.2f} M chars/s "
      f"({len(_sample) / _dt:.1f} cases/s serial)")
print(f"tokens/case: median {np.median(_tok_lens):.0f} | p75 {np.percentile(_tok_lens, 75):.0f} | p90 {np.percentile(_tok_lens, 90):.0f} | "
      f"p99 {np.percentile(_tok_lens, 99):.0f} | max {_tok_lens.max()}")
_budget = MAX_CHUNKS * BODY
print(f"budget {MAX_CHUNKS} x {BODY} = {_budget} tokens: the whole case fits for {100 * (_tok_lens <= _budget).mean():.0f}% of the sample; "
      f"on average the view reads {100 * np.mean(np.minimum(_tok_lens, _budget) / _tok_lens):.0f}% of a case's tokens")
_est_full = len(TRAIN_ALL) / (len(_sample) / _dt)
print(f"serial full-text tokenisation of all train cases would take ~{_est_full / 60:.0f} min; "
      f"DOC_VIEW={DOC_VIEW!r} " + ("tokenises only slices of the text (much faster)." if DOC_VIEW != "spread" else "needs the full text (use N_PROC > 1)."))
if _budget < np.median(_tok_lens): print("note: the budget is below the median case length; that is the point of the view ablation (what part to read).")
_ = _sample, _tok_lens, _chars

sample of 150 train cases | chars/token 4.66 | full-text tokenisation 3.27 M chars/s (159.2 cases/s serial)
tokens/case: median 2752 | p75 4880 | p90 6928 | p99 41351 | max 88617
budget 4 x 511 = 2044 tokens: the whole case fits for 38% of the sample; on average the view reads 70% of a case's tokens
serial full-text tokenisation of all train cases would take ~4 min; DOC_VIEW='tail' tokenises only slices of the text (much faster).
note: the budget is below the median case length; that is the point of the view ablation (what part to read).


## Selecting and tokenising the chunks
Only what the view needs is tokenised: for `tail` / `head` / `head_tail` the notebook tokenises a **character slice** of `CHAR_PER_TOK` characters per wanted token (doubling it if the slice holds too few tokens), which is orders of magnitude cheaper than tokenising a whole judgment. Chunks are aligned to the **end** of the text for the tail (so the last chunk always ends exactly at the end of the case) and to the start for the head. Results are cached on disk and tokenisation runs in `N_PROC` worker processes.

In [9]:
# Cell 15 — Chunk selection (views), tokenisation, cache
def _enc(s):
    return list(tokenizer.encode(s))

def _chunks(ids, from_end):
    """Split a token list into pieces of BODY tokens; alignment from the end (tail) or from the start (head)."""
    n, out = len(ids), []
    if from_end:
        e = n
        while e > 0:
            s = max(0, e - BODY); out.append(ids[s:e]); e = s
        out.reverse()
    else:
        for s in range(0, n, BODY): out.append(ids[s:s + BODY])
    return out

def _slice_tokens(text, need, side):
    """First / last `need` tokens of `text`, tokenising only a character slice. Returns (tokens, text_was_fully_tokenised)."""
    m = need * CHAR_PER_TOK
    while True:
        if m >= len(text):
            ids = _enc(text); return (ids[:need] if side == "head" else ids[-need:]), True
        if side == "head":
            s = text[:m]; j = s.rfind(" "); s = s[:j] if j > m // 2 else s            # drop a cut-off last word
        else:
            s = text[-m:]; j = s.find(" "); s = s[j + 1:] if 0 <= j < m // 2 else s   # drop a cut-off first word
        ids = _enc(s)
        if len(ids) >= need + 8:
            return (ids[:need] if side == "head" else ids[-need:]), False
        m *= 2

def _view_from_full(ids):
    K = MAX_CHUNKS
    if DOC_VIEW == "tail":  return _chunks(ids, True)[-K:]
    if DOC_VIEW == "head":  return _chunks(ids, False)[:K]
    if DOC_VIEW == "head_tail":
        if len(ids) <= K * BODY: return _chunks(ids, False)
        Kh = (K + 1) // 2; Kt = K - Kh
        return _chunks(ids[:Kh * BODY], False) + _chunks(ids[-Kt * BODY:], True)
    P = _chunks(ids, False)                                                          # "spread": K chunks evenly over the whole case
    if len(P) <= K: return P
    return [P[i] for i in sorted(set(np.linspace(0, len(P) - 1, K).round().astype(int).tolist()))]

def select_chunks(text):
    K, n = MAX_CHUNKS, len(text)
    if DOC_VIEW == "spread" or n <= K * BODY * CHAR_PER_TOK:
        pieces = _view_from_full(_enc(text))
    elif DOC_VIEW == "tail":
        ids, _ = _slice_tokens(text, K * BODY, "tail"); pieces = _chunks(ids, True)[-K:]
    elif DOC_VIEW == "head":
        ids, _ = _slice_tokens(text, K * BODY, "head"); pieces = _chunks(ids, False)[:K]
    else:                                                                            # head_tail
        Kh = (K + 1) // 2; Kt = K - Kh
        h, c1 = _slice_tokens(text, Kh * BODY, "head"); t, c2 = _slice_tokens(text, Kt * BODY, "tail")
        pieces = _view_from_full(_enc(text)) if (c1 or c2) else _chunks(h, False)[:Kh] + _chunks(t, True)[-Kt:]
    if not pieces: pieces = [[EOS_ID]]                                               # an empty case still needs one token
    return pieces

_TXT = None
def _work(i):
    return [np.asarray([BOS_ID] + list(p), dtype=np.int32) for p in select_chunks(_TXT[i])]

def tokenize_docs(docs, cache_name):
    """Adds d['chunks'] = list of int32 arrays (each starts with [BOS], length <= CHUNK_TOKENS). Cached on disk."""
    global _TXT
    cdir = WORK_DIR / "cache"; cdir.mkdir(exist_ok=True)
    cp = cdir / f"{cache_name}_v{VOCAB_SIZE}_{DOC_VIEW}_c{CHUNK_TOKENS}_k{MAX_CHUNKS}.pkl"
    cache = pickle.load(open(cp, "rb")) if cp.exists() else {}
    todo = [i for i, d in enumerate(docs) if cache.get(d["uid"], (None, None))[0] != d["nchar"]]
    if todo:
        _TXT = [d["text"] for d in docs]
        res = None
        if N_PROC > 1:
            try:
                with mp.get_context("fork").Pool(N_PROC) as pool:
                    res = list(tqdm(pool.imap(_work, todo, chunksize=8), total=len(todo), desc=f"tokenise {cache_name}", leave=False))
            except Exception as ex:
                print("parallel tokenisation failed (", type(ex).__name__, ex, ") -> serial"); res = None
        if res is None:
            res = [_work(i) for i in tqdm(todo, desc=f"tokenise {cache_name}", leave=False)]
        for i, r in zip(todo, res): cache[docs[i]["uid"]] = (docs[i]["nchar"], r)
        pickle.dump(cache, open(cp, "wb"))
    for d in docs: d["chunks"] = cache[d["uid"]][1]
    return docs

def chunk_stats(docs, title):
    nch = np.asarray([len(d["chunks"]) for d in docs]); ntok = np.asarray([sum(len(c) for c in d["chunks"]) for d in docs])
    print(f"{title}: {len(docs)} cases | chunks/case mean {nch.mean():.2f} (full {MAX_CHUNKS}: {100 * (nch == MAX_CHUNKS).mean():.0f}%) | "
          f"tokens/case read mean {ntok.mean():.0f}")

def slot_offset(n_chunks):
    """Slot of the first chunk. For the tail view the chunks are RIGHT-aligned, so the last chunk (end of the judgment) is always slot K-1."""
    return (MAX_CHUNKS - n_chunks) if DOC_VIEW == "tail" else 0

def build_batch(docs, idxs, with_labels=True):
    """-> ids [N, L], mask [N, L], owner [N] (case position in the batch), slot [N], y [B]."""
    seqs, owner, slot, ys = [], [], [], []
    for b, di in enumerate(idxs):
        d = docs[di]; off = slot_offset(len(d["chunks"]))
        for j, c in enumerate(d["chunks"]):
            seqs.append(c); owner.append(b); slot.append(off + j)
        if with_labels: ys.append(d["y"])
    L = max(len(s) for s in seqs)
    ids = np.full((len(seqs), L), PAD_ID, dtype=np.int64); mask = np.zeros((len(seqs), L), dtype=np.int64)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = s; mask[i, :len(s)] = 1
    y = np.asarray(ys, dtype=np.int64) if with_labels else None
    return ids, mask, np.asarray(owner), np.asarray(slot), y

## Metrics, decision threshold (pure numpy)
Binary macro-F1 = mean of the F1 of class 0 and class 1. The classifier outputs `log p(1) − log p(0)` (the logit margin); predicting `1` when `margin + b > 0` with a tuned `b` is the binary version of the per-class bias tuning of Phase 3. `b` is kept only if a 2-fold, label-stratified check on dev (tune on one half, score the other) shows a real gain.

In [10]:
# Cell 16 — Metrics (macro-F1), confusion, threshold tuning
def bin_scores(y, p):
    y = np.asarray(y, dtype=np.int64); p = np.asarray(p, dtype=np.int64)
    cm = np.zeros((2, 2), dtype=np.int64); np.add.at(cm, (y, p), 1)                  # cm[gold, pred]
    tp = np.diag(cm).astype(np.float64); sup = cm.sum(1).astype(np.float64); npred = cm.sum(0).astype(np.float64)
    P = np.divide(tp, npred, out=np.zeros(2), where=npred > 0)
    R = np.divide(tp, sup,   out=np.zeros(2), where=sup > 0)
    Fs = np.divide(2 * P * R, P + R, out=np.zeros(2), where=(P + R) > 0)
    return dict(P=P, R=R, F=Fs, support=sup, cm=cm, accuracy=float(tp.sum() / max(cm.sum(), 1)),
                macro_f1=float(Fs.mean()), macro_p=float(P.mean()), macro_r=float(R.mean()))

def margin(logp):
    return logp[:, 1] - logp[:, 0]

def evaluate(docs, logp, bias=0.0):
    y = np.asarray([d["y"] for d in docs])
    return bin_scores(y, (margin(logp) + bias > 0).astype(np.int64))

def tune_threshold(docs, logp, grid=np.arange(-3.0, 3.01, 0.1)):
    """Best additive bias b on the margin (ties -> the smallest |b|)."""
    y = np.asarray([d["y"] for d in docs]); m = margin(logp)
    base = bin_scores(y, (m > 0).astype(np.int64))["macro_f1"]; best_b, best = 0.0, base
    for b in sorted(grid.tolist(), key=abs):
        s = bin_scores(y, (m + b > 0).astype(np.int64))["macro_f1"]
        if s > best + 1e-9: best_b, best = float(b), s
    return best_b, base, best

def cv_threshold_gain(docs, logp, folds=2):
    """Honest check: tune on half of the dev cases, score the other half (label-stratified split); mean held-out gain."""
    y = np.asarray([d["y"] for d in docs]); fold = np.zeros(len(docs), dtype=int)
    for c in (0, 1):
        idx = np.where(y == c)[0]; fold[idx] = np.arange(len(idx)) % folds
    gains = []
    for f in range(folds):
        tr, te = np.where(fold != f)[0], np.where(fold == f)[0]
        b, _, _ = tune_threshold([docs[i] for i in tr], logp[tr])
        base = evaluate([docs[i] for i in te], logp[te])["macro_f1"]; tuned = evaluate([docs[i] for i in te], logp[te], b)["macro_f1"]
        gains.append(tuned - base)
    return float(np.mean(gains))

## Model, adaptation methods (LoRMA / LoRA / full / norm-only), training
The bidirectional-attention patch and the LoRA / LoRMA layers are the Phase-3 code, unchanged. New: the **document model** (chunk encoder → chunk pooling → aggregator → head), which replaces the sentence-pooling head.

In [11]:
# Cell 17 — The document model on the Phase-1 encoder, and the adaptation methods (LoRA / LoRMA / IA3)
import copy

# ---- (a) Document model: Phase-1 ENCODER per chunk -> mean-pool -> aggregator over chunks -> Linear(d, 2) -------------------
class DocModel(nn.Module):
    """Every chunk goes through the (adapted) Phase-1 encoder independently; its token vectors are mean-pooled (padding excluded).
    The chunk vectors of a case sit in slots 0..K-1 (right-aligned for the tail view) and are aggregated by AGG."""
    def __init__(self, backbone, d_model, n_slots, agg, dropout, hier_layers):   # backbone = the Phase-1 Encoder
        super().__init__()
        self.backbone, self.agg, self.K = backbone, agg, n_slots
        if agg == "attn":
            self.att_w = nn.Linear(d_model, d_model); self.att_v = nn.Linear(d_model, 1, bias=False)
        elif agg == "hier":
            self.slot_emb = nn.Embedding(n_slots, d_model); nn.init.normal_(self.slot_emb.weight, 0.0, 0.02)
            self.cls = nn.Parameter(torch.zeros(1, 1, d_model)); self.ln_in = nn.LayerNorm(d_model); self.ln_out = nn.LayerNorm(d_model)
            layer = nn.TransformerEncoderLayer(d_model, 8, dim_feedforward=1024, dropout=0.1, activation="gelu", batch_first=True, norm_first=True)
            self.enc = nn.TransformerEncoder(layer, hier_layers, enable_nested_tensor=False)
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(d_model, 2)
        nn.init.normal_(self.head.weight, 0.0, 0.02); nn.init.zeros_(self.head.bias)

    def encode(self, ids, mask):
        """Same computation as the Phase-1 Encoder.forward (bidirectional, key-padding mask), with optional activation checkpointing."""
        enc = self.backbone; N, T = ids.shape                                    # self.backbone is the Phase-1 Encoder here
        x = enc.tok_emb(ids); pos = None
        if enc.pos_emb is not None:
            pos = torch.arange(T, device=ids.device)[None]; x = x + enc.pos_emb(pos)
        x = enc.emb_drop(x)
        for blk in enc.blocks:
            if self.training and GRAD_CKPT:
                x = checkpoint(lambda t, blk=blk: blk(t, rope=enc.rope, key_mask=mask, position_ids=pos), x, use_reentrant=False)
            else:
                x = blk(x, rope=enc.rope, key_mask=mask, position_ids=pos)
        return enc.norm_f(x)

    def forward(self, ids, mask, owner, slot, B):
        h = self.encode(ids, mask).float()                                       # [N, T, d]
        m = mask.unsqueeze(-1).float()
        pooled = (h * m).sum(1) / m.sum(1).clamp_min(1.0)                        # [N, d]   chunk vectors
        D = pooled.size(-1)
        grid = pooled.new_zeros(B, self.K, D); valid = torch.zeros(B, self.K, dtype=torch.bool, device=ids.device)
        grid[owner, slot] = pooled; valid[owner, slot] = True
        alpha = None
        if self.agg == "mean":
            v = valid.unsqueeze(-1).float(); doc = (grid * v).sum(1) / v.sum(1).clamp_min(1.0)
        elif self.agg == "attn":
            s = self.att_v(torch.tanh(self.att_w(grid))).squeeze(-1).masked_fill(~valid, float("-inf"))
            alpha = s.softmax(-1); doc = (alpha.unsqueeze(-1) * grid).sum(1)
        else:
            z = self.ln_in(grid) + self.slot_emb(torch.arange(self.K, device=ids.device)).unsqueeze(0)
            seq = torch.cat([self.cls.expand(B, 1, D), z], dim=1)
            kpm = torch.cat([torch.zeros(B, 1, dtype=torch.bool, device=ids.device), ~valid], dim=1)
            doc = self.ln_out(self.enc(seq, src_key_padding_mask=kpm)[:, 0])
        return self.head(self.drop(doc)), alpha


# ---- (b) LoRA (additive) --------------------------------------------------------------------------------------------------
class LoRALinear(nn.Module):
    """y = W x + (alpha/r) * B(A(dropout(x))).  W frozen; B = 0 at start."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, dropout: float):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale


# ---- (c) LoRMA (multiplicative; Bihany, Patel & Modi, ACL Findings 2025) -----------------------------------------------------
_PI_IDX = {}
def _pi_index(d, dev):
    """idx[i, j] = (j - i) mod d.  gather(M, 1, idx)[i, j] = M[i, (j - i) mod d]  ==  row i of M rotated right by i."""
    k = (d, str(dev))
    if k not in _PI_IDX:
        i = torch.arange(d, device=dev).view(d, 1); j = torch.arange(d, device=dev).view(1, d)
        _PI_IDX[k] = (j - i) % d
    return _PI_IDX[k]

class LoRMALinear(nn.Module):
    """h = I(B A) (W0 x).  W0 frozen.  A in R^{r x d_out}, B in R^{d_out x r}, d_out = base.out_features.
       variant 'plus' (LoRMA+):  I(BA) = I_d + (alpha/r) BA ; computed as y + (alpha/r) B(A y), y = W0 x  -> O(b d r), same order as LoRA.
                                 init B = 0  => exactly the Phase-1 layer at step 0.
       variant 'pi'   (LoRMA_pi): I(BA) = rows of BA rotated cyclically (row i by i); init B[:,0]=1, A[0,0]=1, rest of A = 0
                                 => I(BA) = I_d at step 0 (the rotated 'ones x e0' matrix is the identity)."""
    def __init__(self, base: nn.Linear, r: int, alpha: float, variant: str):
        super().__init__()
        assert variant in ("plus", "pi"), variant
        d = base.out_features
        assert r < d
        self.base, self.variant, self.d, self.scale = base, variant, d, alpha / r
        self.A = nn.Parameter(torch.empty(r, d))
        self.B = nn.Parameter(torch.empty(d, r))
        if variant == "plus":
            nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); nn.init.zeros_(self.B)
        else:
            nn.init.zeros_(self.A)
            nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
            with torch.no_grad():
                self.A[0, 0] = 1.0
                self.B[:, 0] = 1.0

    def forward(self, x):
        y = self.base(x)                                              # W0 x  (frozen), the multiplier acts on the OUTPUT space
        if self.variant == "plus":
            return y + (y @ self.A.t() @ self.B.t()) * self.scale
        M = self.B @ self.A                                           # [d, d], rank <= r
        Tm = torch.gather(M, 1, _pi_index(self.d, M.device))          # rank-inflated by the row rotation
        return y @ Tm.t()                                             # (Tm y) for every token


def apply_adapter(module, targets, make):
    """Replace every nn.Linear whose attribute name is in `targets` by make(linear). Returns how many were replaced."""
    n = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in targets:
            setattr(module, name, make(child)); n += 1
        else:
            n += apply_adapter(child, targets, make)
    return n


# ---- (d) IA3 (Liu et al. 2022): learned per-channel scaling of keys, values and the FFN hidden activation -----------------------
class IA3Out(nn.Module):
    """y = l * (W x)   with l in R^{out_features}, initialised to 1.  Rescales the OUTPUT channels (keys / values).
    (Keys are scaled before RoPE rotates them; the relative-position property of RoPE only needs q and k to be rotated, so it survives.)"""
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):
        y = self.base(x)
        return y * self.l.to(y.dtype)


class IA3In(nn.Module):
    """y = W (l * x)   with l in R^{in_features}, initialised to 1.  Used on down_proj: scaling its INPUT is scaling the FFN hidden
    activation silu(gate(x)) * up(x), i.e. the l_ff of the IA3 paper."""
    def __init__(self, base: nn.Linear):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))


def build_model():
    """Fresh copy of the Phase-1 ENCODER every call (several runs in one session do not contaminate each other)."""
    enc = copy.deepcopy(lm.encoder).cpu()
    ref = copy.deepcopy(enc) if METHOD in ("lora", "lorma", "ia3") else None    # untouched copy for the identity check
    model = DocModel(enc, lm_cfg.d_model, MAX_CHUNKS, AGG, HEAD_DROPOUT, HIER_LAYERS)
    hp = HP[METHOD]
    if METHOD == "full":
        pass                                                                    # encoder + embeddings + aggregator/head all trainable
    elif METHOD in ("lora", "lorma", "ia3"):
        for p in enc.parameters(): p.requires_grad = False
        if METHOD == "lora":
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRALinear(l, hp["r"], hp["alpha"], hp["dropout"]))
            print(f"LoRA applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        elif METHOD == "lorma":
            n = apply_adapter(enc, ADAPTER_TARGETS, lambda l: LoRMALinear(l, hp["r"], hp["alpha"], hp["variant"]))
            print(f"LoRMA[{hp['variant']}] applied to {n} Linear layers (r={hp['r']}, alpha={hp['alpha']})")
        else:
            n_out = apply_adapter(enc, tuple(t for s_, t in (("k", "k_proj"), ("v", "v_proj")) if s_ in IA3_SITES), IA3Out)
            n_in = apply_adapter(enc, ("down_proj",), IA3In) if "ff" in IA3_SITES else 0
            print(f"IA3 applied: {n_out} output-scaling vectors (keys/values) + {n_in} FFN-hidden vectors | sites={IA3_SITES}")
        # The adapted encoder must equal the Phase-1 encoder at initialisation (B = 0 for LoRA / LoRMA+, l = 1 for IA3).
        _ids = torch.randint(5, VOCAB_SIZE, (1, 48)); _m = torch.ones_like(_ids)
        with torch.no_grad():
            a = enc.eval()(_ids, attention_mask=_m); b = ref.eval()(_ids, attention_mask=_m)
        err = (a - b).abs().max().item()
        print(f"identity-at-init check: max|adapted - Phase-1| = {err:.2e}")
        assert err < 1e-4, "adapter does not start from the Phase-1 function"
        enc.train(); del ref
    elif METHOD == "norm_only":
        for p in enc.parameters(): p.requires_grad = False
        n = 0
        for m in enc.modules():
            if isinstance(m, (RMSNorm, LayerNorm)):
                for p in m.parameters(): p.requires_grad = True; n += p.numel()
        print(f"norm-only: {n:,} gain parameters unfrozen")
    model.to(device)
    tot = sum(p.numel() for p in model.parameters()); tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_new = sum(p.numel() for n_, p in model.named_parameters() if not n_.startswith("backbone."))
    print(f"params: total {tot/1e6:.2f}M | trainable {tr/1e6:.4f}M = {tr:,} ({100*tr/tot:.3f}%) | of which new (aggregator + head) {n_new:,} "
          f"| PEFT part (inside the encoder) {tr - n_new:,}")
    model.n_total, model.n_trainable = tot, tr
    return model


def class_weights(train_docs):
    cnt = np.bincount([d["y"] for d in train_docs], minlength=2).astype(np.float64)
    w = np.ones(2)
    if CLASS_WEIGHT == "sqrt_inv":
        nz = cnt > 0
        w[nz] = np.clip(np.sqrt(cnt.max() / cnt[nz]), 1.0, MAX_CLASS_W)
    return torch.tensor(w, dtype=torch.float32, device=device), cnt

In [12]:
# Cell 18 — Inference: cases -> log-probabilities (+ the attention profile over chunk slots)
@torch.no_grad()
def predict_logp(model, docs, batch_docs=None):
    """-> (logp [n, 2], alpha [n, K] or None, seconds). alpha[i, s] = attention weight of slot s (0 where the slot is empty)."""
    model.eval(); bs = batch_docs or 2 * BATCH_DOCS
    order = sorted(range(len(docs)), key=lambda i: sum(len(c) for c in docs[i]["chunks"]))     # similar lengths together
    out = np.zeros((len(docs), 2), dtype=np.float32); al = np.zeros((len(docs), MAX_CHUNKS), dtype=np.float32)
    t0 = time.time()
    for i in range(0, len(order), bs):
        idxs = order[i:i + bs]
        ids, mask, owner, slot, _ = build_batch(docs, idxs, with_labels=False)
        ids, mask = torch.from_numpy(ids).to(device), torch.from_numpy(mask).to(device)
        owner, slot = torch.from_numpy(owner).to(device), torch.from_numpy(slot).to(device)
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            logits, alpha = model(ids, mask, owner, slot, len(idxs))
        out[idxs] = logits.float().log_softmax(-1).cpu().numpy()
        if alpha is not None: al[idxs] = alpha.float().cpu().numpy()
    if device.type == "cuda": torch.cuda.synchronize()
    return out, (al if model.agg == "attn" else None), time.time() - t0

In [13]:
# Cell 19 — Training loop (AdamW, linear warm-up + cosine, grad accumulation, fp16/bf16 autocast)
def train_doc_model(model, train_docs, val_docs, epochs, max_steps=None):
    hp = HP[METHOD]
    cw, counts = class_weights(train_docs)
    print("class counts:", counts.astype(int).tolist(), "| class weights:", np.round(cw.cpu().numpy(), 3).tolist())
    new_p = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    dec, nodec = [], []
    for n, p in model.named_parameters():
        if n.startswith("backbone.") and p.requires_grad:
            (dec if (p.ndim >= 2 and "tok_emb" not in n) else nodec).append(p)
    groups = [dict(params=new_p, lr=hp["head_lr"], base_lr=hp["head_lr"], weight_decay=0.0)]
    if dec:   groups.append(dict(params=dec,   lr=hp["lr"], base_lr=hp["lr"], weight_decay=hp["wd"]))
    if nodec: groups.append(dict(params=nodec, lr=hp["lr"], base_lr=hp["lr"], weight_decay=0.0))
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.999))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
    trainables = [p for p in model.parameters() if p.requires_grad]

    n_items = len(train_docs)
    steps_per_epoch = math.ceil(math.ceil(n_items / BATCH_DOCS) / ACCUM)
    total = epochs * steps_per_epoch
    if max_steps: total = min(total, max_steps)
    warm = max(1, int(WARMUP_FRAC * total))
    def lr_mult(s):
        if s < warm: return (s + 1) / warm
        prog = (s - warm) / max(1, total - warm)
        return 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))
    print(f"{n_items} cases/epoch | {steps_per_epoch} optimiser steps/epoch (effective batch {BATCH_DOCS * ACCUM}) | total {total} steps")

    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    history, losses, step, done = [], [], 0, False
    best = dict(score=-1.0, epoch=0, state=None)
    t_train = time.time(); n_seen = 0
    for ep in range(1, epochs + 1):
        order = list(range(n_items)); random.Random(SEED + ep).shuffle(order)
        model.train(); t0 = time.time(); micro, ep_loss, ep_n = 0, 0.0, 0
        nb = math.ceil(n_items / BATCH_DOCS)
        for bi in range(nb):
            idxs = order[bi * BATCH_DOCS:(bi + 1) * BATCH_DOCS]
            ids, mask, owner, slot, y = build_batch(train_docs, idxs)
            ids, mask, owner, slot, y = (torch.from_numpy(x).to(device) for x in (ids, mask, owner, slot, y))
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
                logits, _ = model(ids, mask, owner, slot, len(idxs))
            loss = F.cross_entropy(logits.float(), y, weight=cw, label_smoothing=LABEL_SMOOTHING)
            scaler.scale(loss / ACCUM).backward()
            micro += 1; lv = loss.item(); losses.append(lv); ep_loss += lv; ep_n += 1; n_seen += len(idxs)
            if micro % ACCUM == 0 or bi == nb - 1:
                m = lr_mult(step)
                for g in opt.param_groups: g["lr"] = g["base_lr"] * m
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainables, GRAD_CLIP)
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step == 10:
                    per = (time.time() - t0) / 10
                    print(f"  >> speed: {per:.1f} s/step | {BATCH_DOCS * ACCUM / per:.1f} cases/s | projected {per * steps_per_epoch / 60:.1f} min/epoch "
                          f"({per * steps_per_epoch * epochs / 60:.0f} min for {epochs} epochs, plus dev evaluation)")
                if step % 25 == 0:
                    print(f"  ep {ep} step {step}/{total} | loss {np.mean(losses[-25 * ACCUM:]):.4f} | lr x{m:.3f} | {time.time() - t0:.0f}s")
                if max_steps and step >= max_steps: done = True; break
                if TIME_BUDGET_MIN and (time.time() - t_train) / 60 > TIME_BUDGET_MIN:
                    print(f"  TIME_BUDGET_MIN={TIME_BUDGET_MIN} reached -> stopping (the best epoch so far is kept)"); done = True; break
        rec = dict(epoch=ep, train_loss=ep_loss / max(ep_n, 1), seconds=time.time() - t0, steps=step)
        if val_docs:
            lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
            rec.update(val_macro_f1=ev["macro_f1"], val_acc=ev["accuracy"], val_P1=float(ev["P"][1]), val_R1=float(ev["R"][1]),
                       val_frac_pred1=float((lp[:, 1] > lp[:, 0]).mean()))
            if ev["macro_f1"] > best["score"]:
                best = dict(score=ev["macro_f1"], epoch=ep,
                            state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
        print(f"epoch {ep}: " + " | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items() if k != "epoch"))
        history.append(rec)
        if done: break
    stats = dict(train_seconds=time.time() - t_train, steps=step, cases_seen=n_seen,
                 peak_gpu_gb=(torch.cuda.max_memory_allocated() / 2**30) if device.type == "cuda" else 0.0,
                 first_losses=float(np.mean(losses[:20])) if losses else None, last_losses=float(np.mean(losses[-20:])) if losses else None)
    if best["state"] is None:                                    # no validation: keep the final weights
        best = dict(score=None, epoch=history[-1]["epoch"],
                    state={n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad})
    return history, best, stats

In [14]:
# Cell 20 — Build data, build the model, train
seed_everything(SEED)
if SMOKE_TEST:
    _r = random.Random(SEED)
    train_docs = _r.sample(TRAIN_ALL, min(SMOKE_TRAIN_DOCS, len(TRAIN_ALL)))
    val_docs = _r.sample(DEV_ALL, min(SMOKE_VAL_DOCS, len(DEV_ALL)))
elif FINAL_FIT:
    train_docs = TRAIN_ALL + DEV_ALL; val_docs = []
else:
    train_docs = TRAIN_ALL; val_docs = DEV_ALL
_t0 = time.time()
tokenize_docs(train_docs, "train_smoke" if SMOKE_TEST else ("train_dev" if FINAL_FIT else "train"))
if val_docs: tokenize_docs(val_docs, "dev_smoke" if SMOKE_TEST else "dev")
print(f"tokenisation {time.time() - _t0:.0f}s")
chunk_stats(train_docs, "train")
if val_docs: chunk_stats(val_docs, "dev")

if val_docs:                                        # reference points: constant predictions
    for c in (0, 1):
        _ev = evaluate(val_docs, np.log(np.tile(np.eye(2, dtype=np.float32)[c] * 0.998 + 0.001, (len(val_docs), 1))))
        print(f"constant-'{c}' baseline: dev macro-F1 {_ev['macro_f1']:.4f} | accuracy {_ev['accuracy']:.3f}")

model = build_model()
history, best, stats = train_doc_model(model, train_docs, val_docs, epochs=EPOCHS, max_steps=SMOKE_STEPS if SMOKE_TEST else None)
model.load_state_dict(best["state"], strict=False)           # best epoch (or final weights if no validation)
print(f"\nbest epoch {best['epoch']} | dev macro-F1 {best['score']} | train {stats['train_seconds']/60:.1f} min | peak GPU {stats['peak_gpu_gb']:.1f} GB")

tokenise train:   0%|          | 0/37387 [00:00<?, ?it/s]

tokenise dev:   0%|          | 0/1988 [00:00<?, ?it/s]

tokenisation 51s
train: 37387 cases | chunks/case mean 3.42 (full 4: 71%) | tokens/case read mean 1659
dev: 1988 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1949
constant-'0' baseline: dev macro-F1 0.3333 | accuracy 0.500
constant-'1' baseline: dev macro-F1 0.3333 | accuracy 0.500
IA3 applied: 12 output-scaling vectors (keys/values) + 6 FFN-hidden vectors | sites=('k', 'v', 'ff')
identity-at-init check: max|adapted - Phase-1| = 0.00e+00
params: total 35.94M | trainable 0.2788M = 278,786 (0.776%) | of which new (aggregator + head) 264,194 | PEFT part (inside the encoder) 14,592
class counts: [22067, 15320] | class weights: [1.0, 1.2000000476837158]
37387 cases/epoch | 2337 optimiser steps/epoch (effective batch 16) | total 14022 steps
  >> speed: 0.6 s/step | 28.9 cases/s | projected 21.6 min/epoch (129 min for 6 epochs, plus dev evaluation)
  ep 1 step 25/14022 | loss 0.6824 | lr x0.030 | 13s
  ep 1 step 50/14022 | loss 0.6815 | lr x0.059 | 26s
  ep 1 step 75/14

In [15]:
# Cell 21 — Smoke-test verdict (SMOKE_TEST=True only)
if SMOKE_TEST and val_docs:
    drop = stats["last_losses"] / stats["first_losses"]
    lp, _, _ = predict_logp(model, val_docs); ev = evaluate(val_docs, lp)
    frac1 = float((lp[:, 1] > lp[:, 0]).mean())
    print(f"loss {stats['first_losses']:.3f} -> {stats['last_losses']:.3f} (x{drop:.2f}) | dev macro-F1 {ev['macro_f1']:.4f} | share predicted '1': {frac1:.2f}")
    ok = drop < 0.95 and 0.02 < frac1 < 0.98
    print("SMOKE VERDICT:", "OK, the pipeline learns -> run the full job" if ok else
          "NOT OK: loss did not fall clearly or one class dominates; ~150 steps is short for an adapter - compare METHOD='full' before debugging")
    print("Check the '>> speed' line above: it projects the minutes per epoch of the full run; set EPOCHS / MAX_CHUNKS / TIME_BUDGET_MIN accordingly.")
else:
    print("not a smoke run")

not a smoke run


In [16]:
# Cell 22 — Dev analysis: scores, per-class table, confusion, threshold tuning, FP/FN by length, attention profile (full runs with a dev set)
THRESH = 0.0 if THRESH_FINAL is None else float(THRESH_FINAL)
dev_ev = dev_ev_tuned = None; thresh_gain = 0.0; dev_extra = {}
if not SMOKE_TEST and val_docs:
    logp_val, alpha_val, t_inf = predict_logp(model, val_docs)
    dev_ev = evaluate(val_docs, logp_val)
    print(f"===== dev | macro-F1 {dev_ev['macro_f1']:.4f} | macro-P {dev_ev['macro_p']:.4f} | macro-R {dev_ev['macro_r']:.4f} | accuracy {dev_ev['accuracy']:.4f}   <- report the UNTUNED macro-F1")
    for c in (0, 1):
        print(f"  class {c}: support {int(dev_ev['support'][c]):5d} | P {dev_ev['P'][c]:.3f} | R {dev_ev['R'][c]:.3f} | F1 {dev_ev['F'][c]:.3f}")
    print("  confusion [gold x pred]:", dev_ev["cm"].tolist())
    print(f"  inference: {t_inf:.1f}s for {len(val_docs)} cases = {1000 * t_inf / len(val_docs):.0f} ms/case ({MAX_CHUNKS} chunks x {CHUNK_TOKENS} tokens max)")
    dev_extra["infer_ms_per_case"] = 1000 * t_inf / len(val_docs)

    if THRESH_TUNE:
        b_t, s0, s1 = tune_threshold(val_docs, logp_val)
        cv_gain = cv_threshold_gain(val_docs, logp_val) if len(val_docs) >= 20 else 0.0
        thresh_gain = cv_gain
        print(f"threshold tuning on dev: {s0:.4f} -> {s1:.4f} (in-sample gain {s1 - s0:+.4f}, optimistic) | 2-fold held-out gain {cv_gain:+.4f} | bias = {b_t:+.2f}")
        if cv_gain >= THRESH_MIN_GAIN:
            THRESH = b_t; dev_ev_tuned = evaluate(val_docs, logp_val, THRESH)
            print(f"  kept. tuned dev macro-F1 {dev_ev_tuned['macro_f1']:.4f}. For FINAL_FIT=True set  THRESH_FINAL = {THRESH:+.2f}")
        else:
            print(f"  held-out gain below THRESH_MIN_GAIN={THRESH_MIN_GAIN}: threshold NOT used")

    # --- characteristics of the errors: FP / FN by case length (character terciles) ---
    y_v = np.asarray([d["y"] for d in val_docs]); p_v = (margin(logp_val) + THRESH > 0).astype(int)
    nchar_v = np.asarray([d["nchar"] for d in val_docs]); cuts = np.percentile(nchar_v, [33.3, 66.7])
    bucket = np.digitize(nchar_v, cuts); names = ["short", "medium", "long"]
    rows = []
    print("\nerrors by case length (terciles of characters):")
    print(f"  {'bucket':8s} {'cases':>6s} {'median chars':>13s} {'acc':>6s} {'macroF1':>8s} {'FP':>5s} {'FN':>5s} {'frac gold=1':>12s}")
    for bk in range(3):
        sel = bucket == bk
        if sel.sum() == 0: continue
        s = bin_scores(y_v[sel], p_v[sel]); fp = int(((p_v == 1) & (y_v == 0) & sel).sum()); fn = int(((p_v == 0) & (y_v == 1) & sel).sum())
        rows.append(dict(bucket=names[bk], n=int(sel.sum()), median_chars=float(np.median(nchar_v[sel])), acc=s["accuracy"], macro_f1=s["macro_f1"], fp=fp, fn=fn, frac_gold1=float(y_v[sel].mean())))
        print(f"  {names[bk]:8s} {int(sel.sum()):6d} {np.median(nchar_v[sel]):13.0f} {s['accuracy']:6.3f} {s['macro_f1']:8.3f} {fp:5d} {fn:5d} {y_v[sel].mean():12.3f}")
    dev_extra["by_length"] = rows
    m_v = margin(logp_val) + THRESH
    dev_extra["mean_margin"] = dict(TP=float(m_v[(y_v == 1) & (p_v == 1)].mean()) if ((y_v == 1) & (p_v == 1)).any() else None,
                                    FP=float(m_v[(y_v == 0) & (p_v == 1)].mean()) if ((y_v == 0) & (p_v == 1)).any() else None,
                                    FN=float(m_v[(y_v == 1) & (p_v == 0)].mean()) if ((y_v == 1) & (p_v == 0)).any() else None,
                                    TN=float(m_v[(y_v == 0) & (p_v == 0)].mean()) if ((y_v == 0) & (p_v == 0)).any() else None)
    print("mean logit margin by outcome (a confident error has a large |margin|):", {k: (None if v is None else round(v, 2)) for k, v in dev_extra["mean_margin"].items()})
    errs = [dict(id=str(val_docs[i]["id"]), gold=int(y_v[i]), pred=int(p_v[i]), margin=float(m_v[i]), nchar=int(nchar_v[i]), n_chunks=len(val_docs[i]["chunks"]))
            for i in np.where(p_v != y_v)[0]]
    (RUN_DIR / "dev_errors.json").write_text(json.dumps(errs))

    # --- which chunk does the model rely on? mean attention weight per slot (AGG='attn') ---
    if alpha_val is not None:
        prof = alpha_val.mean(0); cnt = (alpha_val > 0).mean(0)
        where = {"tail": "slot K-1 = end of the judgment", "head": "slot 0 = start", "head_tail": "first slots = start, last slots = end", "spread": "slot 0 = start ... K-1 = end"}[DOC_VIEW]
        print(f"\nmean attention weight per chunk slot ({where}):", np.round(prof, 3).tolist(), "| share of cases with that slot filled:", np.round(cnt, 2).tolist())
        full = np.asarray([len(d["chunks"]) == MAX_CHUNKS for d in val_docs])
        if full.any(): print("   ... on cases with all slots filled:", np.round(alpha_val[full].mean(0), 3).tolist())
        dev_extra["attn_profile"] = prof.tolist(); dev_extra["attn_profile_full_cases"] = alpha_val[full].mean(0).tolist() if full.any() else None
else:
    print("no dev analysis (smoke run or FINAL_FIT)" + ("" if THRESH_FINAL is None else f" | using THRESH_FINAL={THRESH:+.2f}"))

===== dev | macro-F1 0.6438 | macro-P 0.6579 | macro-R 0.6489 | accuracy 0.6489   <- report the UNTUNED macro-F1
  class 0: support   994 | P 0.620 | R 0.769 | F1 0.686
  class 1: support   994 | P 0.696 | R 0.529 | F1 0.601
  confusion [gold x pred]: [[764, 230], [468, 526]]
  inference: 25.8s for 1988 cases = 13 ms/case (4 chunks x 512 tokens max)
threshold tuning on dev: 0.6438 -> 0.6438 (in-sample gain +0.0000, optimistic) | 2-fold held-out gain +0.0000 | bias = +0.00
  held-out gain below THRESH_MIN_GAIN=0.005: threshold NOT used

errors by case length (terciles of characters):
  bucket    cases  median chars    acc  macroF1    FP    FN  frac gold=1
  short       662          8945  0.680    0.678    62   150        0.532
  medium      664         16700  0.602    0.592    82   182        0.494
  long        662         33629  0.665    0.659    86   136        0.474
mean logit margin by outcome (a confident error has a large |margin|): {'TP': 1.66, 'FP': 0.69, 'FN': -0.91, 'TN': -1.

In [17]:
# Cell 23 — Test predictions -> prediction.json -> submission.zip (full runs only)
def write_submission(docs, logp, bias, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    pred = (margin(logp) + bias > 0).astype(int)
    sub = [dict(id=d["id"], pred=int(p)) for d, p in zip(docs, pred)]            # the id keeps the type it has in test.json
    (out_dir / "prediction.json").write_text(json.dumps(sub, ensure_ascii=False))
    with zipfile.ZipFile(out_dir / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write(out_dir / "prediction.json", arcname="prediction.json")          # must sit at the ROOT of the zip
    print(f"wrote {out_dir/'submission.zip'} | {len(sub)} cases | predicted share of '1': {pred.mean():.3f}")
    verify_submission(out_dir / "submission.zip", docs)
    return sub

def verify_submission(zip_path, docs):
    """Re-open the zip like the evaluator would: root file name, one integer 0/1 per official test id, no extras."""
    zip_path = Path(zip_path)
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        assert names == ["prediction.json"], f"the zip must contain ONLY prediction.json at its root, found {names}"
        sub = json.loads(z.read("prediction.json"))
    assert isinstance(sub, list), "prediction.json must be a JSON list of {id, pred}"
    want = [d["id"] for d in docs]
    got = [e["id"] for e in sub]
    assert len(got) == len(set(map(str, got))), "duplicate ids in the submission"
    assert set(map(str, got)) == set(map(str, want)), f"ids differ from test.json: {len(set(map(str, want)) - set(map(str, got)))} missing, {len(set(map(str, got)) - set(map(str, want)))} unknown"
    assert all(isinstance(e["pred"], int) and not isinstance(e["pred"], bool) and e["pred"] in (0, 1) for e in sub), "pred must be the integer 0 or 1"
    print(f"VERIFIED {zip_path.resolve()} | {len(sub)} cases | first entries {sub[:2]}")
    print("   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).")

if not SMOKE_TEST:
    tokenize_docs(TEST_ALL, "test")
    chunk_stats(TEST_ALL, "test")
    logp_test, _, _ = predict_logp(model, TEST_ALL)
    submission = write_submission(TEST_ALL, logp_test, THRESH, RUN_DIR / "submission")
    _tr = np.bincount([d["y"] for d in train_docs], minlength=2); print(f"train share of '1': {_tr[1] / _tr.sum():.3f}")
else:
    print("smoke run: no submission written")

tokenise test:   0%|          | 0/1517 [00:00<?, ?it/s]

test: 1517 cases | chunks/case mean 3.87 (full 4: 92%) | tokens/case read mean 1947
wrote /kaggle/working/runs/p4_encdec_ia3/submission/submission.zip | 1517 cases | predicted share of '1': 0.382
VERIFIED /kaggle/working/runs/p4_encdec_ia3/submission/submission.zip | 1517 cases | first entries [{'id': '1986_397', 'pred': 0}, {'id': '1977_183', 'pred': 0}]
   -> upload exactly THIS file to the Phase-4 Codabench page (earlier phases produced files with the same name).
train share of '1': 0.410


In [18]:
# Cell 24 — Save trainable weights + the run log (for the report's compute / memory / PEFT-vs-full / representation tables)
def _jsonable(o):
    if isinstance(o, dict): return {str(k): _jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_jsonable(v) for v in o]
    if isinstance(o, np.ndarray): return o.tolist()
    if isinstance(o, (np.floating, np.integer)): return o.item()
    if isinstance(o, Path): return str(o)
    return o

def _ev_summary(ev):
    if ev is None: return None
    return dict(macro_f1=ev["macro_f1"], macro_p=ev["macro_p"], macro_r=ev["macro_r"], accuracy=ev["accuracy"],
                per_class={str(c): dict(support=int(ev["support"][c]), P=round(float(ev["P"][c]), 4), R=round(float(ev["R"][c]), 4), F1=round(float(ev["F"][c]), 4)) for c in (0, 1)},
                confusion=ev["cm"].tolist())

log = dict(run=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
           hier_layers=HIER_LAYERS, final_fit=FINAL_FIT, smoke=SMOKE_TEST, encoder_len=lm_cfg.encoder_len, ia3_sites=list(IA3_SITES), pos_type=lm_cfg.pos_type,
           hp=HP[METHOD], epochs=EPOCHS, batch_docs=BATCH_DOCS, accum=ACCUM, class_weight=CLASS_WEIGHT,
           params_total=model.n_total, params_trainable=model.n_trainable, history=history, best_epoch=best["epoch"], stats=stats,
           threshold=THRESH, threshold_gain=thresh_gain, dev_untuned=_ev_summary(dev_ev), dev_tuned=_ev_summary(dev_ev_tuned), dev_extra=dev_extra,
           n_train=len(train_docs), n_dev=len(val_docs))
(RUN_DIR / "run_log.json").write_text(json.dumps(_jsonable(log), indent=2))
torch.save({"trainable": best["state"], "method": METHOD, "attn_mode": ATTN_MODE, "agg": AGG, "doc_view": DOC_VIEW, "log": _jsonable(log)},
           RUN_DIR / f"cjpe_{RUN_NAME}.pt")
print("saved:", sorted(p.name for p in RUN_DIR.iterdir()))

saved: ['cjpe_p4_encdec_ia3.pt', 'dev_errors.json', 'run_log.json', 'submission']


In [19]:
# Cell 25 — Export this run's document-level distributions ("soft labels") so it can serve as the KD teacher (run #5)
# All four Phase-1 models share one tokenizer, so these arrays can teach ANY student without this notebook's model code.
# Stored as float16 log-probabilities: softmax(logp / T) = softmax(z / T) because logp = z - logsumexp(z).
if EXPORT_SOFT_LABELS and not SMOKE_TEST:
    _t0 = time.time(); _out, _split, _ids = {}, {}, {}
    for _name, _docs in (("train", train_docs), ("dev", val_docs)):
        if not _docs: continue
        _lp, _, _ = predict_logp(model, _docs)
        for d, l in zip(_docs, _lp):
            _out[d["uid"]] = l.astype(np.float16); _split[d["uid"]] = _name; _ids[d["uid"]] = str(d["id"])
    _meta = dict(teacher=RUN_NAME, method=METHOD, attn_mode=ATTN_MODE, doc_view=DOC_VIEW, chunk_tokens=CHUNK_TOKENS, max_chunks=MAX_CHUNKS, agg=AGG,
                 params_total=int(model.n_total), params_trainable=int(model.n_trainable), epochs=int(EPOCHS), best_epoch=int(best["epoch"]),
                 final_fit=FINAL_FIT, dev_macro_f1=None if dev_ev is None else float(dev_ev["macro_f1"]), threshold=float(THRESH))
    _path = RUN_DIR / f"teacher_softlabels_{RUN_NAME}.pt"
    torch.save(dict(meta=_meta, docs=_out, split=_split, ids=_ids), _path)
    print(f"saved {_path} ({_path.stat().st_size/1e6:.1f} MB) in {time.time()-_t0:.0f}s | {len(_out)} cases | teacher params {_meta['params_total']/1e6:.1f}M | dev macro-F1 {_meta['dev_macro_f1']}")
    print("caveat: on the TRAIN cases the teacher is overconfident (it was trained on them); its 'dev' distributions are out-of-sample.")
else:
    print("soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)")

soft-label export skipped (EXPORT_SOFT_LABELS=False or smoke run)


## What to do next

1. **Smoke test first:** `SMOKE_TEST=True` (a few minutes). Read the *masked-LM loss* line (proves the encoder loaded), the *identity-at-init* line (IA³ must equal Phase-1), the trainable-parameter line (IA³ part vs aggregator/head), the **length statistics** (Cell 14), the **`>> speed` line**, the constant-prediction baselines and the smoke verdict. With only ~15k IA³ parameters (+ the document head), 150 steps is short: judge IA³ by the **trend of the loss**, and compare with `METHOD="full"` on the smoke run if in doubt.
2. **Use the same budget as runs #1 and #2** (`DOC_VIEW`, `CHUNK_TOKENS`, `MAX_CHUNKS`, `AGG`), so that the only differences are the backbone (bidirectional MLM-pretrained encoder vs decoders) and the technique. Because the encoder is ~2x cheaper per token than the 12-layer decoders you can afford more epochs; if there is time left, `CHUNK_TOKENS=768` (its trained length) is a legitimate extra control.
3. **Full run:** `SMOKE_TEST=False`. It trains on train, selects the epoch on dev macro-F1, tunes the decision threshold (kept only if it helps on held-out dev halves) and **already writes a valid `submission.zip`**. Upload it to Codabench right away (evaluation takes 15-30 min).
4. **Controls for the report** (same notebook, change Cell 2 only, new `RUN_NAME` each time): `METHOD="lora"` / `"lorma"` (how much do the extra trainable parameters buy?), `METHOD="full"` (can PEFT approach full adaptation?), `IA3_SITES=("k","v")` vs `("ff",)` vs all three (which activations matter?), IA³ learning rate (`lr=1e-2` if it underfits, `1e-3` if unstable), `AGG="mean"` (purest IA³ parameter count). These are for the *analysis*; the 6 required submissions must be different **techniques**, not hyper-parameter variants.
5. **KD:** not as a teacher (35.66M < 36.58M student).
6. Copy `run_log.json` (+ `dev_errors.json`) into the report folder.